# Dockerizing The Pipeline

Dockering means packaging an application (code + dependencies + environment) into a Docker container so it can run consistently anywhere.

In simple terms, it means:

“Putting your project inside Docker so it works the same on every machine.”

let's containerize our pipline. Create a file named ``Dockerfile``:

Dockerfile with pip

```bash
# base Docker image that we will build on
FROM python:3.13.11-slim

# set up our image by installing libraries; pandas in this case
RUN pip install pandas pyarrow

# set up the working directory inside the container
WORKDIR /app
# copy the script to the container. 1st name is source file, 2nd is destination
COPY main.py main.py

# define how to start the script
ENTRYPOINT ["python", "main.py"]
```

FROM: Base image that provides the starting environment for the Docker image.

RUN: Executes commands during the image build process, such as installing dependencies.

WORKDIR: Sets the default working directory inside the container for all subsequent commands.

COPY: Copies files from the host machine into the Docker image.

ENTRYPOINT: Defines the default command that runs automatically when the container starts.


# Let Build and Run Our Dockerfile

**Let's build the image:**

```bash
docker build -t pipeline:pandas .
```
The image name will be pipeline and its tag will be pandas. If the tag isn't specified it will default to latest.

**Lets Run it**
```bash
docker run -it pipeline:pandas 
```
You should get the same output you did when you ran the pipeline script by itself.

Note: these instructions assume that main.py and Dockerfile are in the same directory. The Docker commands should also be run from the same directory as these files.

# Dockerfile With uv

Notice how we used pip ``bash (RUN pip install pandas pyarraow )`` to install the dependencies in above Dockerfile, let change that by using ``uv`` instead

**build and run this**

```bash
# Start with slim Python 3.13 image
FROM python:3.13.10-slim

# Copy uv binary from official uv image (multi-stage build pattern)
COPY --from=ghcr.io/astral-sh/uv:latest /uv /bin/

# Set working directory
WORKDIR /app

# Add virtual environment to PATH so we can use installed packages
ENV PATH="/app/.venv/bin:$PATH"

# Copy dependency files first (better layer caching)
COPY "pyproject.toml" "uv.lock" ".python-version" ./
# Install dependencies from lock file (ensures reproducible builds)
RUN uv sync --locked

# Copy application code
COPY main.py main.py

# Set entry point
ENTRYPOINT ["uv", "run", "python", "main.py"]
```

With this setup, we don’t manually install libraries like `pip install pandas`. Instead, we rely on `pyproject.toml` and `uv.lock` to define and lock all dependencies, and `uv sync --locked` automatically builds the exact same environment inside the Docker image. This ensures reproducible builds because the dependencies are already specified and versioned in the project files.
